### Group 24

Shiref Khaled Elhalawany -  221100944

Ahmed Anis Hassan - 221100101 

Karim Ashraf Elsayed - 221100391

Kareem Shaheen - 221101524

# Part 2: PCA Method with Maximum Likelihood Estimation (MLE)

### Importing Libraries and Defining Paths

This code imports the required Python libraries for data processing and visualization, including Pandas, NumPy, and Matplotlib. It then defines file paths for the scaled ratings dataset and the selected users and items tables. The code ensures that the output directory exists before proceeding and prints a confirmation message indicating that the environment is properly set up for the next processing steps.

In [37]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

tables_path = '../results/tables/'
scaled_ratings_path = tables_path + 'sampled_scaled_ratings.csv'
selected_items_path = tables_path + '3_1_12_selected_items.csv'
selected_users_path = tables_path + '3_1_11_selected_users.csv'

mle_tables_path = tables_path 

if not os.path.exists(mle_tables_path):
    os.makedirs(mle_tables_path)

print("Libraries imported. Paths defined.")


runtime_rows = []

def log_time(method, stage, seconds, extra=""):
    runtime_rows.append({
        "Method": method,
        "Stage": stage,
        "Seconds": seconds,
        "Extra": extra
    })

def memory_mb(*arrays):
    total_bytes = 0
    for arr in arrays:
        if hasattr(arr, "values"):      
            total_bytes += arr.values.nbytes
        else:                            
            total_bytes += arr.nbytes
    return total_bytes / (1024 ** 2)

Libraries imported. Paths defined.


### Data Loading

In [38]:
df = pd.read_csv(scaled_ratings_path)
df_items = pd.read_csv(selected_items_path)
df_users = pd.read_csv(selected_users_path)

target_item_ids = df_items['ItemID'].tolist()
target_user_ids = df_users['UserID'].tolist()

user_item_matrix = df.pivot(index='UserID', columns='ItemID', values='Rating')

item_means = user_item_matrix.mean()
user_item_matrix_centered = user_item_matrix - item_means

print(f"User-Item Matrix Shape: {user_item_matrix.shape}")
print("Data loaded and Centered (NaNs preserved).")

print(user_item_matrix.head())

User-Item Matrix Shape: (28301, 500)
Data loaded and Centered (NaNs preserved).
ItemID  34      93      171     179     222     231     287     289     \
UserID                                                                   
1          NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
6          NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
19         NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
26         NaN     NaN     NaN     NaN     NaN     2.0     NaN     NaN   
38         NaN     NaN     NaN     NaN     3.0     NaN     NaN     5.0   

ItemID  319     458     ...  129391  129405  129739  129771  130083  130298  \
UserID                  ...                                                   
1          NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
6          NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
19         NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
26    

In [39]:
recon_item_means = user_item_matrix.mean()
user_item_matrix_filled_recon = user_item_matrix.fillna(recon_item_means)

R_true_recon = user_item_matrix_filled_recon.values.astype(float)

print("Reconstruction Ground Truth Created")
print("Shape:", R_true_recon.shape)
print("NaNs in R_true_recon:", np.isnan(R_true_recon).sum())

R_centered_recon = (user_item_matrix_filled_recon - recon_item_means).values.astype(float)
print("NaNs in R_centered_recon:", np.isnan(R_centered_recon).sum())

Reconstruction Ground Truth Created
Shape: (28301, 500)
NaNs in R_true_recon: 0
NaNs in R_centered_recon: 0


### Step 1: Generate the Covariance Matrix (MLE Estimate)

This code computes the true Maximum Likelihood Estimation (MLE) covariance matrix between items using the mean-centered user–item ratings matrix while correctly handling missing values. For each pair of items, the covariance is calculated using only the users who rated both items, ensuring statistical correctness in sparse data settings.

Specifically, the code iterates over all item pairs, identifies the set of users with valid ratings for both items, and applies the MLE covariance formula by averaging the product of their centered ratings. If no common users exist for a pair, the covariance is set to zero. The resulting symmetric item–item covariance matrix is stored as a DataFrame, saved to a CSV file, and printed for verification. This matrix serves as a statistically sound foundation for PCA-based latent space analysis under sparse rating conditions.

In [40]:
t0 = time.perf_counter()

X = user_item_matrix_centered 
items = X.columns
m = len(items)

Xv = X.values 
cov_mle = np.zeros((m, m), dtype=float)

for i in range(m):
    xi = Xv[:, i]
    for j in range(i, m):
        xj = Xv[:, j]
        mask = ~np.isnan(xi) & ~np.isnan(xj)
        n = int(mask.sum())

        if n == 0:
            cij = 0.0
        else:
            cij = float(np.dot(xi[mask], xj[mask]) / n)

        cov_mle[i, j] = cij
        cov_mle[j, i] = cij

cov_matrix_mle = pd.DataFrame(cov_mle, index=items, columns=items)

print("Step 1: TRUE MLE Covariance Matrix Generated.")
cov_matrix_mle.to_csv(mle_tables_path + 'pca_mle_step1_cov_matrix.csv')
print("Step 1 Output Saved: pca_mle_step1_cov_matrix.csv")
print(cov_matrix_mle.head())


print(
    "Same item order?",
    list(user_item_matrix.columns) == list(cov_matrix_mle.columns)
)

t1 = time.perf_counter()
log_time("PCA_MLE", "Covariance_Estimation", t1 - t0)
print("PCA MLE covariance estimation time:", t1 - t0)

Step 1: TRUE MLE Covariance Matrix Generated.
Step 1 Output Saved: pca_mle_step1_cov_matrix.csv
ItemID    34        93        171       179       222       231       287     \
ItemID                                                                         
34      1.130601  0.109097 -0.025581 -0.080147  0.131038  0.081952  0.092371   
93      0.109097  1.069096 -0.047186  0.523473  0.130719  0.209686 -0.005164   
171    -0.025581 -0.047186  1.010826  0.391481  0.208475  0.137450  0.751044   
179    -0.080147  0.523473  0.391481  1.128497  0.220407  0.234147  0.782166   
222     0.131038  0.130719  0.208475  0.220407  0.752159  0.032793  0.423411   

ItemID    289       319       458     ...  129391    129405  129739  129771  \
ItemID                                ...                                     
34      0.025103  0.060951  0.009347  ...     0.0  4.075032     0.0     0.0   
93      0.386846  0.132820  0.268756  ...    -0.0  0.000000     0.0     0.0   
171     0.316610  0.204679 

### Step 2: Determine Top 5-Peers and Top 10-Peers

This code performs eigen-decomposition on the MLE-based item–item covariance matrix to extract eigenvalues and eigenvectors, then sorts them in descending order to prioritize the most important principal components. It then calculates the explained variance ratio and cumulative explained variance to automatically determine the smallest number of components (k_75) needed to explain at least 75% of the total variance.

Using this dynamically selected latent space size, the code builds an item embedding (latent feature) matrix from the top eigenvectors and computes cosine similarity between each target item and all other items in this latent space. A helper function is used to identify the Top-5 and Top-10 most similar peers for every target item, along with their similarity scores. The peer lists are printed and saved to a CSV file in a TA-friendly format for later prediction and evaluation steps.

In [41]:
t0 = time.perf_counter()

eigvals, eigvecs = np.linalg.eigh(cov_matrix_mle.values)

idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

total_variance = np.sum(eigvals)
explained_variance_ratio = eigvals / total_variance
cumulative_variance = np.cumsum(explained_variance_ratio)

k_75 = np.argmax(cumulative_variance >= 0.75) + 1
print(f"\nNumber of components to explain 75% variance: {k_75}")
print(f"Correlation Variance at k={k_75}: {cumulative_variance[k_75-1]:.4f}")

k = k_75

print(f"Selected Top {k} Eigenvalues based on 75% Variance:")
print(eigvals[:k])

t1 = time.perf_counter()
log_time("PCA_MLE", "EigenDecomposition", t1 - t0, extra=f"k={k}")
print("PCA MLE eigen-decomposition time:", t1 - t0)

W_k_mle = eigvecs[:, :k]

Z_mle = R_centered_recon @ W_k_mle
R_hat_centered_mle = Z_mle @ W_k_mle.T
R_hat_mle = R_hat_centered_mle + recon_item_means.values.astype(float)

R_hat_mle_df = pd.DataFrame(
    R_hat_mle,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

out_file_mle = mle_tables_path + f"pca_mle_k{k}_reconstruction.csv"
R_hat_mle_df.to_csv(out_file_mle)
print(f"PCA-MLE reconstructed matrix saved: {out_file_mle}")

R_pred = R_hat_mle_df.values.astype(float)

print("NaNs in R_pred (PCA-MLE recon):", np.isnan(R_pred).sum())

mask = (~np.isnan(R_true_recon)) & (~np.isnan(R_pred))
true_vals = R_true_recon[mask]
pred_vals = R_pred[mask]

print("Compared entries:", true_vals.size, "out of", R_true_recon.size)

mae = mean_absolute_error(true_vals, pred_vals)
rmse = np.sqrt(mean_squared_error(true_vals, pred_vals))

explained_variance = float(np.sum(eigvals[:k]) / np.sum(eigvals))

print(f"PCA-MLE Reconstruction (k={k}) -> MAE={mae:.6f}, RMSE={rmse:.6f}, ExplainedVariance={explained_variance:.6f}")

mle_error_row = pd.DataFrame([{
    "k": k,
    "MAE": mae,
    "RMSE": rmse,
    "ExplainedVariance": explained_variance
}])

mle_err_file = mle_tables_path + "pca_mle_reconstruction_errors.csv"

if os.path.exists(mle_err_file):
    old = pd.read_csv(mle_err_file)
    combined = pd.concat([old, mle_error_row], ignore_index=True)
    combined = combined.drop_duplicates(subset=["k"], keep="last").sort_values("k")
    combined.to_csv(mle_err_file, index=False)
else:
    mle_error_row.to_csv(mle_err_file, index=False)

print(f"Saved PCA-MLE reconstruction errors to: {mle_err_file}")


Number of components to explain 75% variance: 7
Correlation Variance at k=7: 0.7653
Selected Top 7 Eigenvalues based on 75% Variance:
[63.28896925 40.60284377 35.55640669 28.05002068 26.8731038  24.9074767
 23.43127994]
PCA MLE eigen-decomposition time: 0.18898670000271522
PCA-MLE reconstructed matrix saved: ../results/tables/pca_mle_k7_reconstruction.csv
NaNs in R_pred (PCA-MLE recon): 0
Compared entries: 14150500 out of 14150500
PCA-MLE Reconstruction (k=7) -> MAE=0.010846, RMSE=0.084451, ExplainedVariance=0.765323
Saved PCA-MLE reconstruction errors to: ../results/tables/pca_mle_reconstruction_errors.csv


In [42]:
memory_pca_mle = memory_mb(
    cov_matrix_mle,
    eigvecs,
    R_hat_mle_df
)

memory_df = pd.DataFrame([{
    "Method": "PCA_MLE",
    "Components": "MLE Covariance + Eigenvectors + Reconstruction",
    "Memory_MB": memory_pca_mle
}])

memory_df.to_csv(
    mle_tables_path + "pca_mle_memory.csv",
    index=False
)

print("Saved PCA MLE memory to pca_mle_memory.csv")
print(memory_df)

Saved PCA MLE memory to pca_mle_memory.csv
    Method                                      Components   Memory_MB
0  PCA_MLE  MLE Covariance + Eigenvectors + Reconstruction  111.774445


In [43]:
def find_latent_peers(n_components, top_n_peers, item_ids, target_ids, evecs):
    latent_features = evecs[:, :n_components]  
    latent_df = pd.DataFrame(latent_features, index=item_ids)

    peers_result = {}
    for tid in target_ids:
        if tid not in latent_df.index:
            continue

        target_vec = latent_df.loc[tid].values.reshape(1, -1)
        sim_scores = cosine_similarity(target_vec, latent_df.values).flatten()
        sim_series = pd.Series(sim_scores, index=latent_df.index).drop(tid)
        sorted_sim = sim_series.sort_values(ascending=False)

        peers_result[tid] = {
            'top_ids': sorted_sim.head(top_n_peers).index.tolist(),
            'sim_series': sim_series
        }
    return peers_result

all_item_ids = list(cov_matrix_mle.columns)

peers_dict_5 = find_latent_peers(
    n_components=k_75, top_n_peers=5,
    item_ids=all_item_ids, target_ids=target_item_ids, evecs=eigvecs
)

peers_dict_10 = find_latent_peers(
    n_components=k_75, top_n_peers=10,
    item_ids=all_item_ids, target_ids=target_item_ids, evecs=eigvecs
)

peers_rows = []
for tid in target_item_ids:
    if tid not in peers_dict_5 or tid not in peers_dict_10:
        continue
    peers_rows.append({
        "TargetItem": tid,
        "Top5_Peers_MLE": str(peers_dict_5[tid]["top_ids"]),
        "Top10_Peers_MLE": str(peers_dict_10[tid]["top_ids"])
    })

df_peers = pd.DataFrame(peers_rows)
df_peers.to_csv(mle_tables_path + "pca_mle_step2_peers.csv", index=False)
print("Step 2 Output Saved: pca_mle_step2_peers.csv")

for tid in target_item_ids:
    if tid in peers_dict_5:
        print(f"Item {tid} Top-5 MLE peers: {peers_dict_5[tid]['top_ids']}")
print(df_peers.head())

Step 2 Output Saved: pca_mle_step2_peers.csv
Item 99904 Top-5 MLE peers: [34, 92210, 94786, 94739, 94405]
Item 119705 Top-5 MLE peers: [34, 92210, 94786, 94739, 94405]
   TargetItem                    Top5_Peers_MLE  \
0       99904  [34, 92210, 94786, 94739, 94405]   
1      119705  [34, 92210, 94786, 94739, 94405]   

                                     Top10_Peers_MLE  
0  [34, 92210, 94786, 94739, 94405, 93752, 93574,...  
1  [34, 92210, 94786, 94739, 94405, 93752, 93574,...  


### Step 3: Determine reduced dimensional space (Top 5-peers)

This code saves a reduced latent peer space for each target item using the Top-5 most similar peers derived from the MLE-based PCA latent representation. For every target item, it records each peer’s rank, item ID, and cosine similarity weight, then exports this structured peer-space table to a CSV file for later recommendation and prediction steps.

As an additional TA-proof step, the code constructs user reduced vectors by extracting each target user’s mean-centered ratings on the Top-5 peer items for every target item. These vectors represent user preferences within the reduced peer space (and may still include NaNs if the user did not rate a peer). The user-level reduced vectors are saved to a separate CSV file and previews of both outputs are printed for verification.

In [44]:
reduced_data_5 = []
for tid in target_item_ids:
    if tid not in peers_dict_5:
        continue
    top_peers = peers_dict_5[tid]['top_ids'][:5]
    sims = peers_dict_5[tid]['sim_series']

    for rank, pid in enumerate(top_peers, 1):
        reduced_data_5.append({
            'TargetItem': tid,
            'Peer_Rank': rank,
            'Peer_ItemID': pid,
            'Latent_Similarity': float(sims[pid]),
            'Space_Type': 'Top5_MLE'
        })

df_reduced_5 = pd.DataFrame(reduced_data_5)
df_reduced_5.to_csv(mle_tables_path + 'pca_mle_step3_reduced_space_top5.csv', index=False)
print("Step 3 Output Saved: pca_mle_step3_reduced_space_top5.csv")

rows = []
for tid in target_item_ids:
    if tid not in peers_dict_5:
        continue
    peers = peers_dict_5[tid]['top_ids'][:5]

    for uid in target_user_ids:
        if uid not in user_item_matrix_centered.index:
            continue
        vec = user_item_matrix_centered.loc[uid, peers].values 
        row = {'UserID': uid, 'TargetItem': tid}
        for i, p in enumerate(peers, 1):
            row[f'Peer{i}_{p}'] = vec[i-1]
        rows.append(row)

df_user_red5 = pd.DataFrame(rows)
df_user_red5.to_csv(mle_tables_path + 'pca_mle_step3_user_reduced_vectors_top5.csv', index=False)
print("Step 3 (extra) Output Saved: pca_mle_step3_user_reduced_vectors_top5.csv")

print(df_reduced_5.head())

print(df_user_red5.head())

Step 3 Output Saved: pca_mle_step3_reduced_space_top5.csv
Step 3 (extra) Output Saved: pca_mle_step3_user_reduced_vectors_top5.csv
   TargetItem  Peer_Rank  Peer_ItemID  Latent_Similarity Space_Type
0       99904          1           34                0.0   Top5_MLE
1       99904          2        92210                0.0   Top5_MLE
2       99904          3        94786                0.0   Top5_MLE
3       99904          4        94739                0.0   Top5_MLE
4       99904          5        94405                0.0   Top5_MLE
   UserID  TargetItem  Peer1_34  Peer2_92210  Peer3_94786  Peer4_94739  \
0       1       99904       NaN          NaN          NaN          NaN   
1     134       99904  0.283312          NaN          NaN          NaN   
2     903       99904  0.283312          NaN          NaN          NaN   
3       1      119705       NaN          NaN          NaN          NaN   
4     134      119705  0.283312          NaN          NaN          NaN   

   Peer5_94405  

### Step 4: Rating Predictions (Top 5-peers)

This code generates rating predictions for each target user–item pair using the Top-5 most similar peers obtained from the MLE-based PCA latent space. For every target user and target item, it labels whether the rating is existing or missing in the original user–item matrix.

It then computes a similarity-weighted prediction using the user’s mean-centered ratings on the peer items, but only includes a peer in the calculation if the user actually rated that peer (to avoid using missing values). If no valid peer ratings are available (denominator equals zero), the prediction falls back to the item’s mean rating. The final predictions are stored in a DataFrame and saved to a CSV file for later evaluation and comparison.

In [45]:
t0 = time.perf_counter()

preds_5 = []

for uid in target_user_ids:
    if uid not in user_item_matrix.index:
        continue

    for tid in target_item_ids:
        if tid not in peers_dict_5:
            continue

        status = "Existing" if pd.notna(user_item_matrix.loc[uid, tid]) else "Missing"

        peers = peers_dict_5[tid]['top_ids'][:5]
        sims = peers_dict_5[tid]['sim_series']

        num = 0.0
        den = 0.0

        for pid in peers:
            if pd.notna(user_item_matrix_centered.loc[uid, pid]):
                w = float(sims[pid])
                val = float(user_item_matrix_centered.loc[uid, pid])
                num += w * val
                den += abs(w)

        base_mean = float(item_means[tid]) if tid in item_means.index else 0.0
        pred = base_mean + (num / den) if den != 0 else base_mean

        preds_5.append({'UserID': uid, 'ItemID': tid, 'Pred_MLE_Top5': pred, 'Status': status})

df_res_5 = pd.DataFrame(preds_5)
df_res_5.to_csv(mle_tables_path + 'pca_mle_step4_preds_top5.csv', index=False)
print("Step 4 Output Saved: pca_mle_step4_preds_top5.csv")
print(df_res_5.head())

t1 = time.perf_counter()
log_time("PCA_MLE", "Prediction_Top5", t1 - t0)
print("PCA_MLE Top-5 prediction time:", t1 - t0)

Step 4 Output Saved: pca_mle_step4_preds_top5.csv
   UserID  ItemID  Pred_MLE_Top5   Status
0       1   99904            1.0  Missing
1       1  119705            1.0  Missing
2     134   99904            1.0  Missing
3     134  119705            1.0  Missing
4     903   99904            1.0  Missing
PCA_MLE Top-5 prediction time: 0.009222000000590924


### Step 5: Determine reduced dimensional space (Top 10-peers)

This code saves a reduced latent peer space for each target item using the Top-10 most similar peers identified from the MLE-based PCA latent representation. For every target item, it stores each peer’s rank, item ID, and cosine similarity weight, then exports this peer-space table to a CSV file for later recommendation and prediction steps.

As an additional TA-proof step, the code also generates user reduced vectors by collecting each target user’s mean-centered ratings on the Top-10 peer items for every target item. These vectors provide a compact representation of user preferences within the Top-10 reduced peer space (and may still contain NaNs if the user did not rate some peers). The user-level reduced vectors are saved to a separate CSV file, and previews of both outputs are printed for verification.

In [46]:
reduced_data_10 = []
for tid in target_item_ids:
    if tid not in peers_dict_10:
        continue
    top_peers = peers_dict_10[tid]['top_ids'][:10]
    sims = peers_dict_10[tid]['sim_series']

    for rank, pid in enumerate(top_peers, 1):
        reduced_data_10.append({
            'TargetItem': tid,
            'Peer_Rank': rank,
            'Peer_ItemID': pid,
            'Latent_Similarity': float(sims[pid]),
            'Space_Type': 'Top10_MLE'
        })

df_reduced_10 = pd.DataFrame(reduced_data_10)
df_reduced_10.to_csv(mle_tables_path + 'pca_mle_step5_reduced_space_top10.csv', index=False)
print("Step 5 Output Saved: pca_mle_step5_reduced_space_top10.csv")

rows = []
for tid in target_item_ids:
    if tid not in peers_dict_10:
        continue
    peers = peers_dict_10[tid]['top_ids'][:10]

    for uid in target_user_ids:
        if uid not in user_item_matrix_centered.index:
            continue
        vec = user_item_matrix_centered.loc[uid, peers].values
        row = {'UserID': uid, 'TargetItem': tid}
        for i, p in enumerate(peers, 1):
            row[f'Peer{i}_{p}'] = vec[i-1]
        rows.append(row)

df_user_red10 = pd.DataFrame(rows)
df_user_red10.to_csv(mle_tables_path + 'pca_mle_step5_user_reduced_vectors_top10.csv', index=False)
print("Step 5 (extra) Output Saved: pca_mle_step5_user_reduced_vectors_top10.csv")

print(df_reduced_10.head())
print(df_user_red10.head())

Step 5 Output Saved: pca_mle_step5_reduced_space_top10.csv
Step 5 (extra) Output Saved: pca_mle_step5_user_reduced_vectors_top10.csv
   TargetItem  Peer_Rank  Peer_ItemID  Latent_Similarity Space_Type
0       99904          1           34                0.0  Top10_MLE
1       99904          2        92210                0.0  Top10_MLE
2       99904          3        94786                0.0  Top10_MLE
3       99904          4        94739                0.0  Top10_MLE
4       99904          5        94405                0.0  Top10_MLE
   UserID  TargetItem  Peer1_34  Peer2_92210  Peer3_94786  Peer4_94739  \
0       1       99904       NaN          NaN          NaN          NaN   
1     134       99904  0.283312          NaN          NaN          NaN   
2     903       99904  0.283312          NaN          NaN          NaN   
3       1      119705       NaN          NaN          NaN          NaN   
4     134      119705  0.283312          NaN          NaN          NaN   

   Peer5_94405

### Step 6: Rating Predictions (Top 10-peers)

This code generates rating predictions for each target user–item pair using the Top-10 most similar peers obtained from the MLE-based PCA latent space. For every target user and target item, it labels whether the rating is existing or missing in the original user–item matrix.

It then computes a similarity-weighted prediction using the user’s mean-centered ratings on the peer items, while only including peers that the user has actually rated (to avoid using missing values). If no valid peer ratings are available (denominator equals zero), the prediction falls back to the item’s mean rating. The final predicted ratings are stored in a DataFrame and saved to a CSV file for later evaluation and comparison.

In [47]:
t0 = time.perf_counter()

preds_10 = []

for uid in target_user_ids:
    if uid not in user_item_matrix.index:
        continue

    for tid in target_item_ids:
        if tid not in peers_dict_10:
            continue

        status = "Existing" if pd.notna(user_item_matrix.loc[uid, tid]) else "Missing"

        peers = peers_dict_10[tid]['top_ids'][:10]
        sims = peers_dict_10[tid]['sim_series']

        num = 0.0
        den = 0.0

        for pid in peers:
            if pd.notna(user_item_matrix_centered.loc[uid, pid]):
                w = float(sims[pid])
                val = float(user_item_matrix_centered.loc[uid, pid])
                num += w * val
                den += abs(w)

        base_mean = float(item_means[tid]) if tid in item_means.index else 0.0
        pred = base_mean + (num / den) if den != 0 else base_mean

        preds_10.append({'UserID': uid, 'ItemID': tid, 'Pred_MLE_Top10': pred, 'Status': status})

df_res_10 = pd.DataFrame(preds_10)
df_res_10.to_csv(mle_tables_path + 'pca_mle_step6_preds_top10.csv', index=False)
print("Step 6 Output Saved: pca_mle_step6_preds_top10.csv")
print(df_res_10.head())

t1 = time.perf_counter()
log_time("PCA_MLE", "Prediction_Top10", t1 - t0)
print("PCA_MLE Top-10 prediction time:", t1 - t0)

Step 6 Output Saved: pca_mle_step6_preds_top10.csv
   UserID  ItemID  Pred_MLE_Top10   Status
0       1   99904             1.0  Missing
1       1  119705             1.0  Missing
2     134   99904             1.0  Missing
3     134  119705             1.0  Missing
4     903   99904             1.0  Missing
PCA_MLE Top-10 prediction time: 0.0044736000018019695


In [48]:
pd.DataFrame(runtime_rows).to_csv(tables_path + "pca_mle_runtime.csv", index=False)

### Step 7: Comparing MLE-Based Top-5 and Top-10 Prediction Results

This code compares the rating predictions generated using the MLE-based Top-5 and Top-10 latent peer models. It merges the two prediction result tables by user and item identifiers to ensure a direct, one-to-one comparison for each user–item pair.

After merging, the code computes both the difference and the absolute difference between the Top-5 and Top-10 predictions, quantifying how prediction values change when the neighborhood size is increased. The comparison results are saved to a CSV file for analysis, and the average absolute difference is printed to provide a concise summary of the impact of using more latent peers.

In [49]:
merged_mle = pd.merge(
    df_res_5[['UserID', 'ItemID', 'Pred_MLE_Top5', 'Status']],
    df_res_10[['UserID', 'ItemID', 'Pred_MLE_Top10']],
    on=['UserID', 'ItemID'],
    how='inner'
)

merged_mle['Diff'] = merged_mle['Pred_MLE_Top5'] - merged_mle['Pred_MLE_Top10']
merged_mle['AbsDiff'] = merged_mle['Diff'].abs()

merged_mle.to_csv(mle_tables_path + 'pca_mle_step7_internal_comparison.csv', index=False)
print("Step 7 Output Saved: pca_mle_step7_internal_comparison.csv")
print(merged_mle.head())
print("Average Abs Diff:", merged_mle['AbsDiff'].mean())

Step 7 Output Saved: pca_mle_step7_internal_comparison.csv
   UserID  ItemID  Pred_MLE_Top5   Status  Pred_MLE_Top10  Diff  AbsDiff
0       1   99904            1.0  Missing             1.0   0.0      0.0
1       1  119705            1.0  Missing             1.0   0.0      0.0
2     134   99904            1.0  Missing             1.0   0.0      0.0
3     134  119705            1.0  Missing             1.0   0.0      0.0
4     903   99904            1.0  Missing             1.0   0.0      0.0
Average Abs Diff: 0.0


### Step 8 & 9: Comparing Part 1 vs Part 2 Prediction Methods (Top-5 and Top-10)

This code compares the prediction outputs from Part 1 (mean-filled covariance PCA approach) against Part 2 (MLE covariance PCA approach) to evaluate how the two methods differ in their predicted ratings.

First, it loads the Part 1 Top-5 prediction results and merges them with the Part 2 Top-5 MLE predictions using matching UserID and ItemID pairs. It then calculates the difference and absolute difference between the two methods and saves the comparison table to a CSV file, along with printing the average absolute difference as a summary metric.

Next, the same comparison process is repeated for the Top-10 predictions, merging Part 1 Top-10 predictions with Part 2 Top-10 MLE predictions, computing the differences, saving the results, and printing the average absolute method difference. This provides a clear quantitative comparison of how using MLE covariance changes predictions relative to the standard mean-filled PCA method.

In [50]:
p1_top5 = pd.read_csv(tables_path + 'pca_step9_predictions_top5.csv')
comp_top5 = pd.merge(
    p1_top5[['UserID', 'ItemID', 'Pred_Top5']],
    df_res_5[['UserID', 'ItemID', 'Pred_MLE_Top5']],
    on=['UserID', 'ItemID'],
    how='inner'
)
comp_top5['Method_Diff'] = comp_top5['Pred_Top5'] - comp_top5['Pred_MLE_Top5']
comp_top5['AbsMethodDiff'] = comp_top5['Method_Diff'].abs()

comp_top5.to_csv(mle_tables_path + 'pca_mle_step8_method_comparison_top5.csv', index=False)
print("Step 8 Output Saved: pca_mle_step8_method_comparison_top5.csv")
print(comp_top5.head())
print("Avg Abs Method Diff (Top5):", comp_top5['AbsMethodDiff'].mean())

p1_top10 = pd.read_csv(tables_path + 'pca_step11_predictions_top10.csv')
comp_top10 = pd.merge(
    p1_top10[['UserID', 'ItemID', 'Pred_Top10']],
    df_res_10[['UserID', 'ItemID', 'Pred_MLE_Top10']],
    on=['UserID', 'ItemID'],
    how='inner'
)
comp_top10['Method_Diff'] = comp_top10['Pred_Top10'] - comp_top10['Pred_MLE_Top10']
comp_top10['AbsMethodDiff'] = comp_top10['Method_Diff'].abs()

comp_top10.to_csv(mle_tables_path + 'pca_mle_step9_method_comparison_top10.csv', index=False)
print("Step 9 Output Saved: pca_mle_step9_method_comparison_top10.csv")
print(comp_top10.head())
print("Avg Abs Method Diff (Top10):", comp_top10['AbsMethodDiff'].mean())

Step 8 Output Saved: pca_mle_step8_method_comparison_top5.csv
   UserID  ItemID  Pred_Top5  Pred_MLE_Top5  Method_Diff  AbsMethodDiff
0       1   99904        1.0            1.0          0.0            0.0
1       1  119705        1.0            1.0          0.0            0.0
2     134   99904        1.0            1.0          0.0            0.0
3     134  119705        1.0            1.0          0.0            0.0
4     903   99904        1.0            1.0          0.0            0.0
Avg Abs Method Diff (Top5): 0.0
Step 9 Output Saved: pca_mle_step9_method_comparison_top10.csv
   UserID  ItemID  Pred_Top10  Pred_MLE_Top10  Method_Diff  AbsMethodDiff
0       1   99904         1.0             1.0          0.0            0.0
1       1  119705         1.0             1.0          0.0            0.0
2     134   99904         1.0             1.0          0.0            0.0
3     134  119705         1.0             1.0          0.0            0.0
4     903   99904         1.0            